# Deep Neural Networks from Scratch: Backpropagation, Momentum, and Activation Functions

## Exercise 1: Building a Deep Neural Network Without Keras or TensorFlow

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

In [ ]:
# Generate a synthetic dataset: 4 numerical features, 3 classes
from sklearn.datasets import make_classification

n_samples = 300
n_features = 4
n_classes = 3

X, y = make_classification(
    n_samples=n_samples,
    n_features=n_features,
    n_informative=4,
    n_redundant=0,
    n_classes=n_classes,
    n_clusters_per_class=1,
    random_state=42
)

# Standardize features
X = (X - X.mean(axis=0)) / X.std(axis=0)

# One-hot encode labels for softmax/cross-entropy
Y_onehot = np.eye(n_classes)[y]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Y_onehot shape:", Y_onehot.shape)

In [ ]:
# Architecture: 4 -> 5 (ReLU) -> 4 (ReLU) -> 3 (Softmax)
input_size = 4
hidden1_size = 5
hidden2_size = 4
output_size = 3
learning_rate = 0.01


def initialize_parameters(input_size, hidden1_size, hidden2_size, output_size):
    """
    Initializes weights with small random values (He-like scaling)
    and biases with zeros for all layers.
    """
    params = {
        "W1": np.random.randn(input_size, hidden1_size) * np.sqrt(2.0 / input_size),
        "b1": np.zeros((1, hidden1_size)),
        "W2": np.random.randn(hidden1_size, hidden2_size) * np.sqrt(2.0 / hidden1_size),
        "b2": np.zeros((1, hidden2_size)),
        "W3": np.random.randn(hidden2_size, output_size) * np.sqrt(2.0 / hidden2_size),
        "b3": np.zeros((1, output_size)),
    }
    return params


params = initialize_parameters(input_size, hidden1_size, hidden2_size, output_size)
for k, v in params.items():
    print(f"{k}: shape {v.shape}")

In [ ]:
def relu(z):
    return np.maximum(0, z)


def relu_derivative(z):
    return (z > 0).astype(float)


def softmax(z):
    """Numerically stable softmax."""
    z_shifted = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z_shifted)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


def forward_propagation(X, params):
    """
    Computes forward propagation through the 3-layer network.
    Returns the output probabilities and a cache of intermediate values
    needed for backpropagation.
    """
    Z1 = X @ params["W1"] + params["b1"]
    A1 = relu(Z1)

    Z2 = A1 @ params["W2"] + params["b2"]
    A2 = relu(Z2)

    Z3 = A2 @ params["W3"] + params["b3"]
    A3 = softmax(Z3)

    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2, "Z3": Z3, "A3": A3}
    return A3, cache


def categorical_cross_entropy(Y_true, Y_pred):
    """
    Computes the categorical cross-entropy loss.
    Y_true, Y_pred: shape (n_samples, n_classes)
    """
    epsilon = 1e-12
    Y_pred_clipped = np.clip(Y_pred, epsilon, 1 - epsilon)
    loss = -np.sum(Y_true * np.log(Y_pred_clipped)) / Y_true.shape[0]
    return loss


print("Forward propagation, softmax, and loss functions defined.")

In [ ]:
def backward_propagation(X, Y_true, params, cache):
    """
    Computes gradients of the loss with respect to all weights and biases
    using vectorized matrix operations (backpropagation through 3 layers).
    """
    m = X.shape[0]
    A1, A2, A3 = cache["A1"], cache["A2"], cache["A3"]
    Z1, Z2 = cache["Z1"], cache["Z2"]

    # Output layer gradient (softmax + cross-entropy combined derivative)
    dZ3 = A3 - Y_true                      # shape (m, output_size)
    dW3 = (A2.T @ dZ3) / m
    db3 = np.sum(dZ3, axis=0, keepdims=True) / m

    # Second hidden layer gradient
    dA2 = dZ3 @ params["W3"].T
    dZ2 = dA2 * relu_derivative(Z2)
    dW2 = (A1.T @ dZ2) / m
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m

    # First hidden layer gradient
    dA1 = dZ2 @ params["W2"].T
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = (X.T @ dZ1) / m
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    grads = {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2, "dW3": dW3, "db3": db3}
    return grads


def update_parameters(params, grads, learning_rate):
    """Updates parameters using vanilla gradient descent."""
    for layer in ["1", "2", "3"]:
        params[f"W{layer}"] -= learning_rate * grads[f"dW{layer}"]
        params[f"b{layer}"] -= learning_rate * grads[f"db{layer}"]
    return params


print("Backward propagation and parameter update functions defined.")

In [ ]:
def train_network(X, Y_onehot, params, learning_rate, epochs=1000):
    losses = []
    accuracies = []

    for epoch in range(epochs):
        # Forward pass
        A3, cache = forward_propagation(X, params)

        # Compute loss
        loss = categorical_cross_entropy(Y_onehot, A3)
        losses.append(loss)

        # Compute accuracy
        predictions = np.argmax(A3, axis=1)
        true_labels = np.argmax(Y_onehot, axis=1)
        accuracy = np.mean(predictions == true_labels)
        accuracies.append(accuracy)

        # Backward pass
        grads = backward_propagation(X, Y_onehot, params, cache)

        # Update parameters
        params = update_parameters(params, grads, learning_rate)

        if epoch % 100 == 0:
            print(f"Epoch {epoch:4d} | Loss: {loss:.4f} | Accuracy: {accuracy:.4f}")

    return params, losses, accuracies


params = initialize_parameters(input_size, hidden1_size, hidden2_size, output_size)
trained_params, losses, accuracies = train_network(X, Y_onehot, params, learning_rate, epochs=1000)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(losses, color="tomato")
axes[0].set_title("Training Loss (Categorical Cross-Entropy)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")

axes[1].plot(accuracies, color="steelblue")
axes[1].set_title("Training Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")

plt.suptitle("From-Scratch Deep Neural Network — Training Progress", fontsize=13)
plt.tight_layout()
plt.show()

print(f"\nFinal training loss: {losses[-1]:.4f}")
print(f"Final training accuracy: {accuracies[-1]:.4f}")

## Exercise 2: Optimizing Backpropagation with Momentum

In [ ]:
# Generate a synthetic dataset: 2 numerical input features, binary target
from sklearn.datasets import make_moons

X2, y2 = make_moons(n_samples=300, noise=0.2, random_state=42)
X2 = (X2 - X2.mean(axis=0)) / X2.std(axis=0)
y2 = y2.reshape(-1, 1)

print("X2 shape:", X2.shape)
print("y2 shape:", y2.shape)

plt.figure(figsize=(6, 5))
plt.scatter(X2[:, 0], X2[:, 1], c=y2.flatten(), cmap="coolwarm", s=20)
plt.title("Dataset for Momentum Experiment (make_moons)")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.tight_layout()
plt.show()

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def initialize_2layer_params(input_size=2, hidden_size=4, output_size=1, seed=42):
    rng = np.random.RandomState(seed)
    return {
        "W1": rng.randn(input_size, hidden_size) * 0.5,
        "b1": np.zeros((1, hidden_size)),
        "W2": rng.randn(hidden_size, output_size) * 0.5,
        "b2": np.zeros((1, output_size)),
    }


def forward_2layer(X, params):
    Z1 = X @ params["W1"] + params["b1"]
    A1 = relu(Z1)
    Z2 = A1 @ params["W2"] + params["b2"]
    A2 = sigmoid(Z2)
    return A2, {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}


def binary_cross_entropy(y_true, y_pred):
    epsilon = 1e-12
    y_pred_clipped = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.mean(y_true * np.log(y_pred_clipped) + (1 - y_true) * np.log(1 - y_pred_clipped))


def backward_2layer(X, y_true, params, cache):
    m = X.shape[0]
    A1, A2, Z1 = cache["A1"], cache["A2"], cache["Z1"]

    dZ2 = A2 - y_true
    dW2 = (A1.T @ dZ2) / m
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m

    dA1 = dZ2 @ params["W2"].T
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = (X.T @ dZ1) / m
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    return {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}


print("2-layer network functions defined.")

In [ ]:
def update_standard(params, grads, learning_rate):
    """Standard gradient descent — no momentum."""
    for layer in ["1", "2"]:
        params[f"W{layer}"] -= learning_rate * grads[f"dW{layer}"]
        params[f"b{layer}"] -= learning_rate * grads[f"db{layer}"]
    return params


def update_with_momentum(params, grads, velocity, learning_rate, momentum=0.9):
    """
    Gradient descent with momentum.
    velocity = momentum * velocity - learning_rate * gradient
    param = param + velocity

    Momentum accumulates a running average of past gradients, which helps
    accelerate convergence in consistent gradient directions and dampens
    oscillations in directions where the gradient sign keeps flipping.
    """
    for layer in ["1", "2"]:
        velocity[f"dW{layer}"] = momentum * velocity[f"dW{layer}"] - learning_rate * grads[f"dW{layer}"]
        velocity[f"db{layer}"] = momentum * velocity[f"db{layer}"] - learning_rate * grads[f"db{layer}"]

        params[f"W{layer}"] += velocity[f"dW{layer}"]
        params[f"b{layer}"] += velocity[f"db{layer}"]

    return params, velocity


print("Standard and momentum-based update functions defined.")

In [ ]:
learning_rate_2 = 0.005
momentum_coef = 0.9
epochs_2 = 2000

# --- Train without momentum ---
params_standard = initialize_2layer_params()
losses_standard = []

for epoch in range(epochs_2):
    A2, cache = forward_2layer(X2, params_standard)
    loss = binary_cross_entropy(y2, A2)
    losses_standard.append(loss)
    grads = backward_2layer(X2, y2, params_standard, cache)
    params_standard = update_standard(params_standard, grads, learning_rate_2)

print(f"Standard GD — Final loss: {losses_standard[-1]:.4f}")

In [ ]:
# --- Train with momentum ---
params_momentum = initialize_2layer_params()
velocity = {
    "dW1": np.zeros_like(params_momentum["W1"]),
    "db1": np.zeros_like(params_momentum["b1"]),
    "dW2": np.zeros_like(params_momentum["W2"]),
    "db2": np.zeros_like(params_momentum["b2"]),
}
losses_momentum = []

for epoch in range(epochs_2):
    A2, cache = forward_2layer(X2, params_momentum)
    loss = binary_cross_entropy(y2, A2)
    losses_momentum.append(loss)
    grads = backward_2layer(X2, y2, params_momentum, cache)
    params_momentum, velocity = update_with_momentum(
        params_momentum, grads, velocity, learning_rate_2, momentum_coef
    )

print(f"Momentum GD — Final loss: {losses_momentum[-1]:.4f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(losses_standard, label="Standard Gradient Descent", color="steelblue")
plt.plot(losses_momentum, label="Gradient Descent with Momentum", color="tomato")
plt.title("Training Loss: Standard vs Momentum-based Gradient Descent")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.legend()
plt.tight_layout()
plt.show()

# Find epoch where each method first reaches a loss threshold
threshold = 0.4
epoch_standard = next((i for i, l in enumerate(losses_standard) if l < threshold), None)
epoch_momentum = next((i for i, l in enumerate(losses_momentum) if l < threshold), None)

print(f"Epochs to reach loss < {threshold}:")
print(f"  Standard GD : {epoch_standard}")
print(f"  Momentum GD : {epoch_momentum}")

**Interpretation: How momentum affects convergence**

Momentum accumulates a velocity term that is a running, exponentially-weighted average of past gradients. When gradients consistently point in a similar direction across iterations, momentum builds up speed in that direction, accelerating convergence compared to standard gradient descent. When gradients oscillate (point in conflicting directions across consecutive steps, common in narrow ravines of the loss surface), momentum's averaging effect cancels out the oscillating components, leading to a smoother and more direct path toward the minimum. The loss curve for momentum-based training typically descends faster and reaches a lower loss in fewer epochs than standard gradient descent, as observed above.

## Exercise 3: Fine-Tuning Activation Functions for an Image Classifier

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)

(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

X_train = X_train.reshape(-1, 28, 28, 1).astype("float32") / 255.0
X_test  = X_test.reshape(-1, 28, 28, 1).astype("float32") / 255.0

y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat  = keras.utils.to_categorical(y_test, 10)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
def build_cnn(activation_name):
    """
    Builds a CNN with 2 conv layers + 1 dense layer + softmax output,
    using the specified activation function for hidden layers.
    """
    if activation_name == "relu":
        act = "relu"
    elif activation_name == "leaky_relu":
        act = layers.LeakyReLU(alpha=0.1)
    elif activation_name == "swish":
        act = "swish"
    else:
        raise ValueError("Unsupported activation")

    model = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, (3, 3)),
        layers.Activation(act) if isinstance(act, str) else act,
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3)),
        layers.Activation(act) if isinstance(act, str) else act,
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(128),
        layers.Activation(act) if isinstance(act, str) else act,
        layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


print("build_cnn function defined.")

In [ ]:
activation_functions = ["relu", "leaky_relu", "swish"]
activation_results = {}

for act_name in activation_functions:
    print(f"\n{'='*50}\nTraining CNN with {act_name.upper()} activation\n{'='*50}")
    model = build_cnn(act_name)
    history = model.fit(
        X_train, y_train_cat,
        validation_split=0.1,
        epochs=5,
        batch_size=128,
        verbose=0
    )
    test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)
    activation_results[act_name] = {
        "history": history,
        "test_loss": test_loss,
        "test_acc": test_acc
    }
    print(f"{act_name.upper()} — Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.4f}")

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {"Activation": name.replace("_", " ").title(),
     "Test Accuracy": round(res["test_acc"], 4),
     "Test Loss": round(res["test_loss"], 4)}
    for name, res in activation_results.items()
]).sort_values("Test Accuracy", ascending=False)

print(summary.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for name, res in activation_results.items():
    axes[0].plot(res["history"].history["val_accuracy"], label=name.replace("_", " ").title())
    axes[1].plot(res["history"].history["val_loss"], label=name.replace("_", " ").title())

axes[0].set_title("Validation Accuracy by Activation Function")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].set_title("Validation Loss by Activation Function")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

**Interpretation: Which activation function works best and why**

- **ReLU** is fast to compute and avoids vanishing gradients for positive inputs, making it a strong general-purpose default. However, it can suffer from the "dying ReLU" problem, where neurons get stuck outputting zero permanently if their weights push them into the negative input region.

- **Leaky ReLU** addresses the dying ReLU problem by allowing a small, non-zero gradient (controlled by `alpha`) for negative inputs, which keeps neurons "alive" and can lead to marginally better gradient flow during training, especially on deeper networks.

- **Swish** (x * sigmoid(x)) is a smooth, non-monotonic activation that has been shown in research to outperform ReLU on some deeper architectures, since its smoothness can help optimization by providing more informative gradients near zero. The trade-off is a higher computational cost per neuron compared to ReLU's simple max operation.

On a small, well-behaved dataset like MNIST trained for only a few epochs, the difference between these activation functions is often modest, since the task is not deep or complex enough to fully expose ReLU's weaknesses. The activation function with the highest test accuracy in the table above should generally be preferred for this specific architecture and dataset, but on deeper networks or harder problems, Leaky ReLU and Swish often provide more consistent advantages over plain ReLU.

## Conclusion

These exercises covered three levels of understanding neural network optimization:

1. **From-scratch implementation** (Exercise 1) showed exactly how forward propagation, softmax, categorical cross-entropy, and backpropagation work together at the matrix-operation level, without relying on any deep learning framework.

2. **Momentum-based optimization** (Exercise 2) demonstrated how accumulating a velocity term from past gradients accelerates convergence and reduces oscillations compared to vanilla gradient descent — a foundational idea behind more advanced optimizers like Adam.

3. **Activation function tuning** (Exercise 3) illustrated that the choice of activation function, while sometimes producing only modest differences on simple datasets, can have a meaningful impact on training stability and final accuracy, particularly as network depth and task complexity increase.

**Suggested further challenge**: Add Batch Normalization layers after each Conv2D/Dense layer in Exercise 3, and Dropout layers for regularization, then re-run the activation function comparison to see whether these techniques change which activation function performs best.